##### **`Python 3, using Conda env: orthanc`**
##### **`@Author : Hasan Shaikh`**
##### **`@Email  : hasanshaikh3198@gmail.com`**
##### **`@GitHub : https://github.com/hash123shaikh`**

---

In [ ]:
"""
DICOM Anonymization Script
Compliant with DICOM PS 3.15 Annex E Basic Profile.
Practices adopted from CTP (Clinical Trial Processor) anonymization pipeline.

Anonymization actions used:
  - REPLACE : Tag value replaced with anonymous ID (HN0001, HN0002, ...)
  - EMPTY   : Tag value cleared to empty string; tag structure kept intact
  - REMOVE  : Tag deleted entirely from the dataset
  - HASH    : UID regenerated using pydicom.uid.generate_uid() (deterministic per run)
  - KEEP    : Tag left unchanged (clinically needed for research)

Total tags handled: ~100+
----------------------------------------------------------------
PATIENT IDENTIFIERS (34 tags):
  REPLACE : PatientName, PatientID
  EMPTY   : PatientBirthDate, PatientSex
  REMOVE  : PatientBirthTime, PatientAge, PatientSize, PatientWeight,
            PatientAddress, OtherPatientIDs, OtherPatientNames,
            OtherPatientIDsSeq, PatientBirthName, IssuerOfPatientID,
            InsurancePlanIdentification, PatientMotherBirthName,
            MilitaryRank, BranchOfService, MedicalRecordLocator,
            MedicalAlerts, ContrastAllergies, CountryOfResidence,
            RegionOfResidence, PatientPhoneNumbers, EthnicGroup,
            Occupation, SmokingStatus, AdditionalPatientHistory,
            PregnancyStatus, LastMenstrualDate, PatientReligiousPreference,
            PatientSexNeutered, ResponsiblePerson, ResponsibleOrganization,
            PatientComments, PatientInsurancePlanCodeSeq,
            PatientPrimaryLanguageCodeSeq

INSTITUTIONAL INFO (5 tags):
  REMOVE  : InstitutionName, InstitutionAddress, InstitutionCodeSeq,
            InstitutionalDepartmentName, TimezoneOffsetFromUTC

PERSONNEL (14 tags):
  EMPTY   : ReferringPhysicianName, PerformingPhysicianName,
            OperatorName, PhysicianOfRecord, NameOfPhysicianReadingStudy
  REMOVE  : ReferringPhysicianAddress, ReferringPhysicianPhoneNumbers,
            ReferringPhysiciansIDSeq, PerformingPhysicianIdSeq,
            PhysicianOfRecordIdSeq, OperatorsIdentificationSeq,
            AdmittingDiagnosisDescription, AdmittingDiagnosisCodeSeq,
            PhysicianReadingStudyIdSeq

DEVICE INFO (7 tags):
  EMPTY   : StationName
  REMOVE  : DeviceSerialNumber, PlateID, GeneratorID,
            CassetteID, GantryID, DetectorID

DATES (9 tags):
  EMPTY   : StudyDate, SeriesDate, AcquisitionDate, ContentDate
  REMOVE  : OverlayDate, CurveDate, AcquisitionDatetime,
            OverlayTime, CurveTime

ADMIN IDENTIFIERS (2 tags):
  EMPTY   : AccessionNumber, StudyID

ACQUISITION / PROTOCOL (4 tags):
  REMOVE  : ProtocolName, AcquisitionComments,
            AcquisitionDeviceProcessingDescription,
            AcquisitionProtocolDescription

FREE TEXT / COMMENTS (6 tags):
  REMOVE  : ImageComments, StudyComments, FrameComments,
            IdentifyingComments, DerivationDescription,
            ContributionDescription

SCHEDULING / WORKFLOW (20 tags):
  REMOVE  : ScheduledStationAET, SPSStartDate, SPSStartTime,
            SPSEndDate, SPSEndTime, ScheduledPerformingPhysicianName,
            SPSDescription, ScheduledStationName, SPSLocation,
            PreMedication, PerformedStationAET, PerformedStationName,
            PerformedLocation, PPSStartDate, PPSStartTime, PPSEndDate,
            PPSID, PPSDescription, PPSComments, RequestAttributesSeq

VISIT / ADMISSION (17 tags):
  REMOVE  : AdmissionID, IssuerOfAdmissionID,
            ScheduledAdmissionDate, ScheduledAdmissionTime,
            ScheduledDischargeDate, ScheduledDischargeTime,
            AdmittingDate, AdmittingTime, DischargeDiagnosisDescription,
            SpecialNeeds, ServiceEpisodeID, IssuerOfServiceEpisodeId,
            ServiceEpisodeDescription, CurrentPatientLocation,
            PatientInstitutionResidence, PatientState, VisitComments

ORDER / PROCEDURE (10 tags):
  REMOVE  : ReasonForStudy, RequestingPhysician, RequestingService,
            RequestedProcedureDescription, RequestedProcedureID,
            PatientTransportArrangements, RequestedProcedureLocation,
            RequestedProcedureComments, ImagingServiceRequestComments,
            ConfidentialityPatientData

REFERENCES / SEQUENCES (6 tags):
  REMOVE  : RefStudySeq, RefPPSSeq, RefPatientSeq,
            RefImageSeq, SourceImageSeq, IconImageSequence

DIGITAL SIGNATURES (7 tags):
  REMOVE  : DigitalSignatureUID, RefDigitalSignatureSeq,
            RefSOPInstanceMACSeq, MAC, DigitalSignaturesSeq,
            ModifiedAttributesSequence, OriginalAttributesSequence

UIDs (7 tags):
  HASH    : SOPInstanceUID, StudyInstanceUID, SeriesInstanceUID,
            MediaStorageSOPInstanceUID, FrameOfReferenceUID,
            IrradiationEventUID, ConcatenationUID

COMPLIANCE MARKERS (not PHI, not counted above):
  SET     : PatientIdentityRemoved -> 'YES'
            DeidentificationMethod -> 'DICOM PS 3.15 Annex E - Python Script'
            LongitudinalTemporalInformationModified -> 'REMOVED'
----------------------------------------------------------------
"""

---

### **============ `Case 1: DICOM FILE ANONYMIZATION` ============**

In [10]:
import pydicom
from pydicom.uid import generate_uid
from pathlib import Path
import pandas as pd
from datetime import datetime
import logging


class DICOMAnonymizer:
    """
    Anonymize DICOM files according to DICOM PS 3.15 Annex E guidelines.
    Practices adopted from CTP (Clinical Trial Processor) anonymization pipeline.
    Creates a mapping file for potential future de-anonymization.

    Key differences from basic anonymization:
      - UIDs are regenerated (not kept) to prevent cross-study re-identification
      - Extended patient demographics (SmokingStatus, PregnancyStatus, etc.) removed
      - Scheduling, workflow, visit, admission, and order tags removed
      - Digital signature tags removed (signatures become invalid post-anonymization)
      - Consistent use of EMPTY vs REMOVE:
            EMPTY  -> mandatory DICOM tags where tag must remain for viewer compatibility
            REMOVE -> optional tags where complete deletion is safe

    Expected folder structure:
        input_folder/
            <PatientID>/          <- folder name is the original patient ID
                CT/               <- multiple .dcm files
                RTSTRUCT/         <- single .dcm file
    """

    def __init__(self):
        """
        Initialize anonymizer with sequential ID counter and UID mapping cache.
        No salt required; ID assignment is sequential (HN0001, HN0002, ...).
        """
        self.mapping_records = []
        self._id_counter = 1    # Sequential counter for HN#### ID generation
        self._id_map     = {}   # Maps original PatientID -> anonymous HN#### ID within a run
        self._uid_map    = {}   # Maps original UID -> regenerated UID within a run
        self.setup_logging()

    def setup_logging(self):
        """Setup audit logging to a date-stamped text file."""
        log_filename = f'anonymization_log_{datetime.now().strftime("%Y%m%d")}.txt'
        logging.basicConfig(
            filename=log_filename,
            level=logging.INFO,
            format='%(asctime)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)

    def generate_anonymous_id(self, patient_id: str) -> str:
        """
        Generate a sequential anonymous ID (HN0001, HN0002, ...) from real patient ID.
        The same patient_id always maps to the same anonymous ID within a single run.

        Args:
            patient_id: Original PatientID from DICOM tag

        Returns:
            Anonymous ID string in format HN####
        """
        if patient_id not in self._id_map:
            self._id_map[patient_id] = f"HN{self._id_counter:04d}"
            self._id_counter += 1
        return self._id_map[patient_id]

    def hash_uid(self, original_uid: str) -> str:
        """
        Regenerate a UID deterministically using pydicom.uid.generate_uid().
        The same original UID always maps to the same new UID within a single run,
        preserving cross-file references (e.g. StudyInstanceUID consistent across
        all series files belonging to the same study).

        Args:
            original_uid: Original UID string from DICOM tag

        Returns:
            New anonymized UID string; returns original if empty/blank
        """
        if not original_uid or original_uid.strip() == '':
            return original_uid
        if original_uid not in self._uid_map:
            self._uid_map[original_uid] = generate_uid()
        return self._uid_map[original_uid]

    # ── Helper methods (CTP action equivalents) ──────────────────────────────

    def _empty(self, ds, tag_name: str):
        """
        Clear tag value to empty string while keeping tag structure intact.
        Equivalent to CTP @empty() action.
        Used for mandatory DICOM tags where complete removal may break viewer
        compatibility (e.g. PatientBirthDate, AccessionNumber).
        """
        if hasattr(ds, tag_name):
            setattr(ds, tag_name, '')

    def _remove(self, ds, tag_name: str):
        """
        Delete tag entirely from the dataset.
        Equivalent to CTP @remove() action.
        Used for optional tags where complete deletion is safe and preferred.
        """
        if hasattr(ds, tag_name):
            delattr(ds, tag_name)

    def _hash_uid_tag(self, ds, tag_name: str):
        """
        Regenerate UID tag with a new anonymized UID.
        Equivalent to CTP @hashuid() action.
        Consistent mapping ensures cross-file UID references remain valid.
        """
        if hasattr(ds, tag_name):
            original = str(getattr(ds, tag_name))
            setattr(ds, tag_name, self.hash_uid(original))

    # ─────────────────────────────────────────────────────────────────────────

    def anonymize_dicom_file(self, input_path: Path, output_path: Path) -> bool:
        """
        Anonymize a single DICOM file and save to output path.

        Args:
            input_path:  Path to original DICOM file
            output_path: Path where anonymized file will be saved

        Returns:
            True if successful, False otherwise
        """
        try:
            ds = pydicom.dcmread(input_path)

            # ── Store original values before modification (for mapping CSV) ──
            original_patient_id   = str(ds.get('PatientID',        'UNKNOWN'))
            original_patient_name = str(ds.get('PatientName',      'UNKNOWN'))
            original_dob          = str(ds.get('PatientBirthDate',  ''))
            original_sex          = str(ds.get('PatientSex',        ''))

            # Generate sequential anonymous ID; reuses existing ID if patient already seen
            anon_id = self.generate_anonymous_id(original_patient_id)

            # Save mapping record once per unique patient
            if not any(r['anonymous_id'] == anon_id for r in self.mapping_records):
                self.mapping_records.append({
                    'anonymous_id':          anon_id,
                    'original_patient_id':   original_patient_id,
                    'original_patient_name': original_patient_name,
                    'original_dob':          original_dob,
                    'original_sex':          original_sex,
                    'anonymization_date':    datetime.now().isoformat()
                })

            # ============ PATIENT IDENTIFIERS (34 tags) ============

            # PatientName and PatientID: always present; replace directly
            ds.PatientName = anon_id
            ds.PatientID   = anon_id

            # Core demographics: empty (keep tag, clear value) for viewer compatibility
            self._empty(ds, 'PatientBirthDate')                 # Keep tag; clear value
            self._empty(ds, 'PatientSex')                       # Keep tag; clear value

            # Extended demographics: remove entirely
            self._remove(ds, 'PatientBirthTime')
            self._remove(ds, 'PatientAge')                      # Derived from DOB; remove
            self._remove(ds, 'PatientSize')
            self._remove(ds, 'PatientWeight')
            self._remove(ds, 'PatientAddress')
            self._remove(ds, 'IssuerOfPatientID')               # Added from CTP; links to issuing authority
            self._remove(ds, 'OtherPatientIDs')                 # Alternative hospital IDs
            self._remove(ds, 'OtherPatientNames')               # Aliases / maiden names
            self._remove(ds, 'OtherPatientIDsSeq')              # Added from CTP; sequence form of above
            self._remove(ds, 'PatientBirthName')                # Added from CTP
            self._remove(ds, 'InsurancePlanIdentification')     # Added from CTP
            self._remove(ds, 'PatientMotherBirthName')          # Added from CTP
            self._remove(ds, 'MilitaryRank')                    # Added from CTP
            self._remove(ds, 'BranchOfService')                 # Added from CTP
            self._remove(ds, 'MedicalRecordLocator')            # Added from CTP; links to hospital record
            self._remove(ds, 'MedicalAlerts')                   # Added from CTP
            self._remove(ds, 'ContrastAllergies')               # Added from CTP
            self._remove(ds, 'CountryOfResidence')              # Added from CTP
            self._remove(ds, 'RegionOfResidence')               # Added from CTP
            self._remove(ds, 'PatientPhoneNumbers')             # Added from CTP
            self._remove(ds, 'EthnicGroup')
            self._remove(ds, 'Occupation')
            self._remove(ds, 'SmokingStatus')                   # Added from CTP
            self._remove(ds, 'AdditionalPatientHistory')
            self._remove(ds, 'PregnancyStatus')                 # Added from CTP
            self._remove(ds, 'LastMenstrualDate')               # Added from CTP
            self._remove(ds, 'PatientReligiousPreference')      # Added from CTP
            self._remove(ds, 'PatientSexNeutered')              # Added from CTP
            self._remove(ds, 'ResponsiblePerson')               # Added from CTP
            self._remove(ds, 'ResponsibleOrganization')         # Added from CTP
            self._remove(ds, 'PatientComments')
            self._remove(ds, 'PatientInsurancePlanCodeSeq')     # Added from CTP
            self._remove(ds, 'PatientPrimaryLanguageCodeSeq')   # Added from CTP

            # ============ INSTITUTIONAL INFO (5 tags) ============
            self._remove(ds, 'InstitutionName')                 # CTP removes; prior version replaced with 'ANONYMIZED'
            self._remove(ds, 'InstitutionAddress')
            self._remove(ds, 'InstitutionCodeSeq')              # Added from CTP; coded form of institution
            self._remove(ds, 'InstitutionalDepartmentName')
            self._remove(ds, 'TimezoneOffsetFromUTC')           # Added from CTP; can narrow geographic location

            # ============ PERSONNEL (14 tags) ============

            # Mandatory DICOM tags: empty (keep tag, clear value)
            self._empty(ds, 'ReferringPhysicianName')
            self._empty(ds, 'PerformingPhysicianName')
            self._empty(ds, 'PhysicianOfRecord')
            self._empty(ds, 'NameOfPhysicianReadingStudy')

            # Optional personnel tags: remove entirely
            self._remove(ds, 'OperatorName')
            self._remove(ds, 'ReferringPhysicianAddress')
            self._remove(ds, 'ReferringPhysicianPhoneNumbers')
            self._remove(ds, 'ReferringPhysiciansIDSeq')        # Added from CTP; sequence form
            self._remove(ds, 'PerformingPhysicianIdSeq')        # Added from CTP; sequence form
            self._remove(ds, 'PhysicianOfRecordIdSeq')          # Added from CTP; sequence form
            self._remove(ds, 'OperatorsIdentificationSeq')      # Added from CTP; sequence form
            self._remove(ds, 'AdmittingDiagnosisDescription')   # Added from CTP; clinical info
            self._remove(ds, 'AdmittingDiagnosisCodeSeq')       # Added from CTP; coded form of above
            self._remove(ds, 'PhysicianReadingStudyIdSeq')      # Added from CTP; sequence form

            # ============ DEVICE INFO (7 tags) ============
            # Device identifiers can be traced back to the scanner and indirectly to the institution
            self._empty(ds, 'StationName')                      # Empty rather than remove for viewer compatibility
            self._remove(ds, 'DeviceSerialNumber')
            self._remove(ds, 'PlateID')                         # CR-specific; safe to remove for CT
            self._remove(ds, 'GeneratorID')                     # Added from CTP
            self._remove(ds, 'CassetteID')                      # Added from CTP
            self._remove(ds, 'GantryID')
            self._remove(ds, 'DetectorID')

            # ============ DATES (9 tags) ============
            # All acquisition-related dates removed; temporal information not
            # required for this study. If date intervals are needed between
            # timepoints, implement date-shifting (incrementdate) instead.
            self._empty(ds, 'StudyDate')                        # Empty for viewer compatibility
            self._empty(ds, 'SeriesDate')                       # Empty for viewer compatibility
            self._empty(ds, 'AcquisitionDate')                  # Empty for viewer compatibility
            self._empty(ds, 'ContentDate')                      # Empty for viewer compatibility
            self._remove(ds, 'OverlayDate')                     # Added from CTP; optional overlay tag
            self._remove(ds, 'CurveDate')                       # Added from CTP; retired DICOM tag
            self._remove(ds, 'AcquisitionDatetime')             # Added from CTP; combined date-time form
            self._remove(ds, 'OverlayTime')                     # Added from CTP
            self._remove(ds, 'CurveTime')                       # Added from CTP; retired DICOM tag

            # ============ ADMIN IDENTIFIERS (2 tags) ============
            # AccessionNumber links back to RIS/HIS and can re-identify the patient;
            # empty rather than remove for DICOM structural integrity
            self._empty(ds, 'AccessionNumber')
            self._empty(ds, 'StudyID')

            # ============ ACQUISITION / PROTOCOL (4 tags) ============
            self._remove(ds, 'ProtocolName')
            self._remove(ds, 'AcquisitionComments')
            self._remove(ds, 'AcquisitionDeviceProcessingDescription')  # Added from CTP
            self._remove(ds, 'AcquisitionProtocolDescription')          # Added from CTP

            # ============ FREE TEXT / COMMENTS (8 tags) ============
            # Free-text fields may contain patient names, diagnoses, or other PHI
            self._remove(ds, 'ImageComments')
            self._remove(ds, 'StudyComments')
            self._remove(ds, 'FrameComments')                   # Added from CTP
            self._remove(ds, 'IdentifyingComments')             # Added from CTP; legacy tag
            self._remove(ds, 'DerivationDescription')           # Added from CTP; may contain PHI
            self._remove(ds, 'ContributionDescription')         # Added from CTP
            self._remove(ds, 'StudyDescription')                # ADD THIS — contains institution name (CMC)
            self._remove(ds, 'SeriesDescription')               # ADD THIS — contains TPS name (ARIA RadOnc)

            # ============ RTSTRUCT-SPECIFIC TAGS (4 tags) ============
            # Only present in RTSTRUCT files; _remove/_empty silently skip if absent in CT
            self._remove(ds, 'ReviewerName')            # (300e,0008) Reviewer identity
            self._empty(ds,  'StructureSetLabel')       # (3006,0002) May contain patient/institution info
            self._empty(ds,  'StructureSetName')        # (3006,0004) May contain patient/institution info
            self._remove(ds, 'StructureSetDescription') # (3006,0006) Free text; PHI risk

            # ============ SCHEDULING / WORKFLOW (20 tags) ============
            # Added from CTP; scheduling tags carry procedure, location, and personnel info
            self._remove(ds, 'ScheduledStationAET')
            self._remove(ds, 'SPSStartDate')
            self._remove(ds, 'SPSStartTime')
            self._remove(ds, 'SPSEndDate')
            self._remove(ds, 'SPSEndTime')
            self._remove(ds, 'ScheduledPerformingPhysicianName')
            self._remove(ds, 'SPSDescription')
            self._remove(ds, 'ScheduledStationName')
            self._remove(ds, 'SPSLocation')
            self._remove(ds, 'PreMedication')
            self._remove(ds, 'PerformedStationAET')
            self._remove(ds, 'PerformedStationName')
            self._remove(ds, 'PerformedLocation')
            self._remove(ds, 'PPSStartDate')
            self._remove(ds, 'PPSStartTime')
            self._remove(ds, 'PPSEndDate')
            self._remove(ds, 'PPSID')
            self._remove(ds, 'PPSDescription')
            self._remove(ds, 'PPSComments')
            self._remove(ds, 'RequestAttributesSeq')            # Sequence containing SPS attributes

            # ============ VISIT / ADMISSION (17 tags) ============
            # Added from CTP; admission and visit tags carry patient location and
            # clinical history that could re-identify the patient
            self._remove(ds, 'AdmissionID')
            self._remove(ds, 'IssuerOfAdmissionID')
            self._remove(ds, 'ScheduledAdmissionDate')
            self._remove(ds, 'ScheduledAdmissionTime')
            self._remove(ds, 'ScheduledDischargeDate')
            self._remove(ds, 'ScheduledDischargeTime')
            self._remove(ds, 'AdmittingDate')
            self._remove(ds, 'AdmittingTime')
            self._remove(ds, 'DischargeDiagnosisDescription')
            self._remove(ds, 'SpecialNeeds')
            self._remove(ds, 'ServiceEpisodeID')
            self._remove(ds, 'IssuerOfServiceEpisodeId')
            self._remove(ds, 'ServiceEpisodeDescription')
            self._remove(ds, 'CurrentPatientLocation')          # Ward/bed info
            self._remove(ds, 'PatientInstitutionResidence')
            self._remove(ds, 'PatientState')
            self._remove(ds, 'VisitComments')

            # ============ ORDER / PROCEDURE (10 tags) ============
            # Added from CTP; procedure request fields may contain clinical context
            # and requesting physician info that can re-identify the patient
            self._remove(ds, 'ReasonForStudy')
            self._remove(ds, 'RequestingPhysician')
            self._remove(ds, 'RequestingService')
            self._remove(ds, 'RequestedProcedureDescription')
            self._remove(ds, 'RequestedProcedureID')
            self._remove(ds, 'PatientTransportArrangements')
            self._remove(ds, 'RequestedProcedureLocation')
            self._remove(ds, 'RequestedProcedureComments')
            self._remove(ds, 'ImagingServiceRequestComments')
            self._remove(ds, 'ConfidentialityPatientData')

            # ============ REFERENCES / SEQUENCES (6 tags) ============
            # Added from CTP; reference sequences may link back to original studies
            # or patients in the source PACS
            self._remove(ds, 'RefStudySeq')
            self._remove(ds, 'RefPPSSeq')
            self._remove(ds, 'RefPatientSeq')
            self._remove(ds, 'RefImageSeq')
            self._remove(ds, 'SourceImageSeq')
            self._remove(ds, 'IconImageSequence')               # Thumbnail image may contain burned-in PHI

            # ============ DIGITAL SIGNATURES (7 tags) ============
            # Added from CTP; digital signatures become cryptographically invalid
            # after any tag modification, so keeping them is misleading; remove entirely
            self._remove(ds, 'DigitalSignatureUID')
            self._remove(ds, 'RefDigitalSignatureSeq')
            self._remove(ds, 'RefSOPInstanceMACSeq')
            self._remove(ds, 'MAC')
            self._remove(ds, 'DigitalSignaturesSeq')
            self._remove(ds, 'ModifiedAttributesSequence')      # Records prior values before modification
            self._remove(ds, 'OriginalAttributesSequence')      # Records original values before modification

            # ============ PRIVATE TAGS ============
            # Varian ARIA and Siemens CT scanners embed proprietary private tags
            # that frequently contain patient name, ID, and institution info.
            # Remove all private tags entirely.
            ds.remove_private_tags()

            # ============ UIDs (7 tags) ============
            # Regenerate all UIDs deterministically using hash_uid(); the same original
            # UID always maps to the same new UID within a run, preserving cross-file
            # references (e.g. StudyInstanceUID consistent across all series files)
            self._hash_uid_tag(ds, 'SOPInstanceUID')
            self._hash_uid_tag(ds, 'StudyInstanceUID')
            self._hash_uid_tag(ds, 'SeriesInstanceUID')
            self._hash_uid_tag(ds, 'FrameOfReferenceUID')
            self._hash_uid_tag(ds, 'IrradiationEventUID')
            self._hash_uid_tag(ds, 'ConcatenationUID')

            # Update MediaStorageSOPInstanceUID in file meta to match new SOPInstanceUID
            if hasattr(ds, 'file_meta') and hasattr(ds.file_meta, 'MediaStorageSOPInstanceUID'):
                original_media_uid = str(ds.file_meta.MediaStorageSOPInstanceUID)
                ds.file_meta.MediaStorageSOPInstanceUID = self.hash_uid(original_media_uid)

            # ============ COMPLIANCE MARKERS ============
            # Required by DICOM PS 3.15 Annex E to flag de-identified files
            ds.PatientIdentityRemoved = 'YES'
            ds.DeidentificationMethod = 'DICOM PS 3.15 Annex E - Python Script'
            if hasattr(ds, 'LongitudinalTemporalInformationModified'):
                ds.LongitudinalTemporalInformationModified = 'REMOVED'

            # Save anonymized file; create parent directories if they don't exist
            output_path.parent.mkdir(parents=True, exist_ok=True)
            ds.save_as(output_path)

            self.logger.info(
                f"Anonymized: {input_path.name} ({original_patient_name} -> {anon_id})"
            )
            return True

        except Exception as e:
            self.logger.error(f"Error anonymizing {input_path}: {str(e)}")
            return False

    def anonymize_study(self, input_folder: str, output_folder: str,
                        mapping_csv: str = None):
        """
        Anonymize an entire study folder using the expected structure:
            input_folder/
                <PatientID>/          <- folder name is the original patient ID
                    CT/               <- multiple .dcm files
                    RTSTRUCT/         <- single .dcm file

        Output mirrors the same folder structure under output_folder, with
        patient folder names replaced by anonymous IDs (HN0001, HN0002, ...).

        Args:
            input_folder: Path to root folder containing patient subfolders
            output_folder: Path where anonymized files will be saved
            mapping_csv:  Path where original <-> anonymous ID mapping will be
                          saved. Defaults to <output_folder>/patient_mapping.csv
        """
        input_path  = Path(input_folder)
        output_path = Path(output_folder)

        # Default mapping CSV to inside the output folder if not specified
        if mapping_csv is None:
            mapping_csv = str(output_path / 'patient_mapping.csv')

        # Create output directory and mapping CSV parent directory upfront,
        # before any file processing, to avoid FileNotFoundError on to_csv()
        output_path.mkdir(parents=True, exist_ok=True)
        Path(mapping_csv).parent.mkdir(parents=True, exist_ok=True)

        print(f"\nStarting anonymization of: {input_folder}")
        print(f"Output will be saved to:   {output_folder}")
        print(f"Mapping will be saved to:  {mapping_csv}\n")

        # ── Discover patient folders ──────────────────────────────────────
        # Each direct subfolder of input_folder is a patient folder whose
        # name is the original patient ID (e.g. 003262P, 005121C, ...)
        patient_folders = sorted([
            p for p in input_path.iterdir()
            if p.is_dir()
        ])

        if not patient_folders:
            print(f"WARNING: No patient subfolders found under: {input_folder}")
            print(f"  Check that the path is correct and the folder is not empty.")
            pd.DataFrame(self.mapping_records).to_csv(mapping_csv, index=False)
            return

        print(f"Found {len(patient_folders)} patient folders\n")

        total_files   = 0
        total_success = 0

        for patient_folder in patient_folders:
            folder_patient_id = patient_folder.name    # e.g. "003262P"

            # Collect all .dcm files under CT/ and RTSTRUCT/ subfolders
            dicom_files = []
            ct_count     = 0
            rtstruct_count = 0

            for subfolder in ['CT', 'RTSTRUCT']:
                subfolder_path = patient_folder / subfolder
                if subfolder_path.exists():
                    subfolder_files = []
                    for ext in ['*.dcm', '*.DCM']:
                        subfolder_files.extend(subfolder_path.glob(ext))
                    dicom_files.extend(subfolder_files)
                    if subfolder == 'CT':
                        ct_count = len(subfolder_files)
                    else:
                        rtstruct_count = len(subfolder_files)
                else:
                    # Subfolder missing; log and continue gracefully
                    self.logger.warning(
                        f"Expected subfolder not found: {subfolder_path}"
                    )
                    print(f"  WARNING: '{subfolder}' subfolder missing "
                          f"for patient {folder_patient_id}")

            if not dicom_files:
                print(f"  WARNING: No DICOM files found for patient "
                      f"{folder_patient_id} — skipping")
                continue

            print(f"  Patient: {folder_patient_id} "
                  f"({len(dicom_files)} files — "
                  f"CT: {ct_count}, RTSTRUCT: {rtstruct_count})")

            patient_success = 0
            anon_id = None

            for dcm_file in dicom_files:
                # Preserve subfolder structure (CT/ or RTSTRUCT/) under anonymous ID folder
                relative_subfolder = dcm_file.relative_to(patient_folder)  # e.g. CT/filename.dcm

                # Peek at PatientID tag to assign anonymous ID;
                # also cross-checks DICOM tag against folder name
                try:
                    ds_peek        = pydicom.dcmread(dcm_file, stop_before_pixels=True)
                    tag_patient_id = str(ds_peek.get('PatientID', folder_patient_id))
                except Exception:
                    tag_patient_id = folder_patient_id  # Fall back to folder name if unreadable

                # Log a warning if folder name and DICOM tag disagree
                if tag_patient_id != folder_patient_id:
                    self.logger.warning(
                        f"Patient ID mismatch: folder='{folder_patient_id}' "
                        f"vs DICOM tag='{tag_patient_id}' in {dcm_file.name}. "
                        f"Using DICOM tag for anonymization."
                    )

                # Generate anonymous ID; reuses same ID for all files of this patient
                anon_id = self.generate_anonymous_id(tag_patient_id)

                # Output path: output_folder/<anon_id>/CT_or_RTSTRUCT/filename.dcm
                output_file = output_path / anon_id / relative_subfolder

                if self.anonymize_dicom_file(dcm_file, output_file):
                    patient_success += 1

            print(f"    -> Anonymized as {anon_id}: "
                  f"{patient_success}/{len(dicom_files)} files processed")

            total_files   += len(dicom_files)
            total_success += patient_success

        # Save patient mapping to CSV
        mapping_df = pd.DataFrame(self.mapping_records)
        mapping_df.to_csv(mapping_csv, index=False)

        print(f"\n========================================")
        print(f"Anonymization Complete!")
        print(f"  Patients processed:           {len(self.mapping_records)}")
        print(f"  Files successfully processed: {total_success}/{total_files}")
        print(f"  Mapping saved to:             {mapping_csv}")
        print(f"========================================")
        print(f"\nIMPORTANT: Keep '{mapping_csv}' secure and "
              f"separate from anonymized data!")


# ============ USAGE ============
if __name__ == "__main__":
    anonymizer = DICOMAnonymizer()
    anonymizer.anonymize_study(
        input_folder='FOLLOW-UP_PATIENTS_FOR_2_YEARS_LRR__&__NON-LRR',
        output_folder='anonymized_study'
        # mapping_csv defaults to: output_folder/patient_mapping.csv
    )


Starting anonymization of: FOLLOW-UP_PATIENTS_FOR_2_YEARS_LRR__&__NON-LRR
Output will be saved to:   anonymized_study
Mapping will be saved to:  anonymized_study/patient_mapping.csv

Found 102 patient folders

  Patient: 003262P (225 files — CT: 224, RTSTRUCT: 1)
    -> Anonymized as HN0001: 225/225 files processed
  Patient: 005121C (158 files — CT: 157, RTSTRUCT: 1)
    -> Anonymized as HN0002: 158/158 files processed
  Patient: 009644P (232 files — CT: 231, RTSTRUCT: 1)
    -> Anonymized as HN0003: 232/232 files processed
  Patient: 016874P (159 files — CT: 158, RTSTRUCT: 1)
    -> Anonymized as HN0004: 159/159 files processed
  Patient: 020234P (118 files — CT: 117, RTSTRUCT: 1)
    -> Anonymized as HN0005: 118/118 files processed
  Patient: 021617P (142 files — CT: 141, RTSTRUCT: 1)
    -> Anonymized as HN0006: 142/142 files processed
  Patient: 022834D (146 files — CT: 145, RTSTRUCT: 1)
    -> Anonymized as HN0007: 146/146 files processed
  Patient: 027064G (165 files — CT: 164,

---

### **============ `Case 2: PNG FILE ANONYMIZATION` ============**

In [11]:
# ============ PNG FILE ANONYMIZATION ============
# Renames PNG files from original patient ID to anonymous ID (HN0001, etc.)
# using the mapping CSV generated by the DICOM anonymization step.
#
# Expected PNG filename format: <PatientID>_anything.png or <PatientID>.png
# Example: 003262P_dose.png -> HN0001_dose.png
#          005121C.png      -> HN0002.png
#
# Requirements:
#   - mapping_csv must exist (generated by DICOMAnonymizer.anonymize_study)
#   - PNG files must start with the original patient ID as it appears in the mapping

import pandas as pd
import shutil
from pathlib import Path

def anonymize_png_files(
    png_folder:   str,
    output_folder: str,
    mapping_csv:  str,
    copy: bool = True    # True = copy renamed files; False = rename in-place
):
    """
    Rename PNG files by replacing the original patient ID prefix with
    the corresponding anonymous ID (HN0001, HN0002, ...) from the mapping CSV.

    Args:
        png_folder:    Folder containing PNG files named with original patient IDs
        output_folder: Folder where renamed PNG files will be saved
        mapping_csv:   Path to patient_mapping.csv generated by DICOMAnonymizer
        copy:          If True, copies files to output_folder with new names.
                       If False, renames files in-place (png_folder is ignored).
    """
    png_path    = Path(png_folder)
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    # ── Load mapping CSV ──────────────────────────────────────────────────
    mapping_csv_path = Path(mapping_csv)
    if not mapping_csv_path.exists():
        print(f"ERROR: Mapping CSV not found at: {mapping_csv}")
        print(f"  Run DICOMAnonymizer.anonymize_study() first to generate it.")
        return

    mapping_df = pd.read_csv(mapping_csv_path)

    # Validate required columns
    required_cols = {'original_patient_id', 'anonymous_id'}
    if not required_cols.issubset(mapping_df.columns):
        print(f"ERROR: Mapping CSV must contain columns: {required_cols}")
        print(f"  Found columns: {set(mapping_df.columns)}")
        return

    # Build lookup: original_patient_id -> anonymous_id
    # Strip whitespace from both columns to avoid silent mismatches
    id_map = {
        str(row['original_patient_id']).strip(): str(row['anonymous_id']).strip()
        for _, row in mapping_df.iterrows()
    }

    print(f"\nLoaded mapping for {len(id_map)} patients from: {mapping_csv}")
    print(f"Scanning PNG files in: {png_folder}\n")

    # ── Process PNG files ─────────────────────────────────────────────────
    png_files = sorted(png_path.glob('*.png')) + sorted(png_path.glob('*.PNG'))

    if not png_files:
        print(f"WARNING: No PNG files found in: {png_folder}")
        return

    print(f"Found {len(png_files)} PNG files\n")

    success   = 0
    skipped   = 0
    not_found = []

    for png_file in png_files:
        original_stem = png_file.stem    # filename without extension e.g. "003262P_dose"
        original_name = png_file.name    # full filename e.g. "003262P_dose.png"

        # ── Match original patient ID ─────────────────────────────────────
        # Try to match the longest possible patient ID prefix first,
        # then fall back to exact stem match
        matched_original_id = None
        matched_anon_id     = None

        for original_id in id_map:
            # Check if filename starts with this patient ID
            # Accept: 003262P.png, 003262P_anything.png, 003262P-anything.png
            if (original_stem == original_id or
                original_stem.startswith(original_id + '_') or
                original_stem.startswith(original_id + '-')):
                matched_original_id = original_id
                matched_anon_id     = id_map[original_id]
                break

        if matched_original_id is None:
            # No match found in mapping
            not_found.append(original_name)
            skipped += 1
            print(f"  SKIPPED (no mapping found): {original_name}")
            continue

        # ── Build new filename ────────────────────────────────────────────
        # Replace original patient ID prefix with anonymous ID
        # e.g. "003262P_dose.png" -> "HN0001_dose.png"
        #      "003262P.png"      -> "HN0001.png"
        suffix_part  = original_stem[len(matched_original_id):]   # e.g. "_dose" or ""
        new_stem     = matched_anon_id + suffix_part               # e.g. "HN0001_dose"
        new_filename = new_stem + png_file.suffix                  # e.g. "HN0001_dose.png"
        output_file  = output_path / new_filename

        # ── Copy or rename ────────────────────────────────────────────────
        if copy:
            shutil.copy2(png_file, output_file)
        else:
            png_file.rename(output_file)

        print(f"  {original_name:40s} ->  {new_filename}")
        success += 1

    # ── Summary ───────────────────────────────────────────────────────────
    action = "copied" if copy else "renamed"
    print(f"\n========================================")
    print(f"PNG Anonymization Complete!")
    print(f"  Successfully {action}: {success}/{len(png_files)} files")
    print(f"  Skipped (no mapping):  {skipped}/{len(png_files)} files")
    if not_found:
        print(f"\n  Files with no mapping match:")
        for f in not_found:
            print(f"    - {f}")
    print(f"  Output saved to: {output_folder}")
    print(f"========================================")


# ── USAGE ─────────────────────────────────────────────────────────────────────
anonymize_png_files(
    png_folder    = 'data/pre-processed/cmc_c',
    output_folder = 'data/pre-processed/anonymized_cmc_png',
    mapping_csv   = 'anonymized_study/patient_mapping.csv',
    copy          = True    # Set to False to rename in-place instead of copying
)


Loaded mapping for 102 patients from: anonymized_study/patient_mapping.csv
Scanning PNG files in: data/pre-processed/cmc_c

Found 102 PNG files

  003262P.png                              ->  HN0001.png
  005121C.png                              ->  HN0002.png
  009644P.png                              ->  HN0003.png
  016874P.png                              ->  HN0004.png
  020234P.png                              ->  HN0005.png
  021617P.png                              ->  HN0006.png
  022834D.png                              ->  HN0007.png
  027064G.png                              ->  HN0008.png
  033839P.png                              ->  HN0009.png
  035486D.png                              ->  HN0010.png
  039932P.png                              ->  HN0011.png
  061461H.png                              ->  HN0012.png
  061741P.png                              ->  HN0013.png
  068574P.png                              ->  HN0014.png
  075733P.png                             

---

### ============ **`CSV FILE ANONYMIZATION`** ============

In [12]:
# ============ CSV FILE ANONYMIZATION ============
# Replaces original patient IDs in a specified column with anonymous IDs
# (HN0001, HN0002, ...) using the mapping CSV generated by DICOMAnonymizer.
#
# All other columns are left completely unchanged.
#
# Requirements:
#   - mapping_csv must exist (generated by DICOMAnonymizer.anonymize_study)
#   - The ID column name must match exactly as it appears in the CSV header

import pandas as pd
from pathlib import Path

def anonymize_csv_id_column(
    input_csv:    str,
    output_csv:   str,
    mapping_csv:  str,
    id_column:    str    # Exact column name containing the original patient IDs
):
    """
    Replace original patient IDs in a single column of a CSV file with
    anonymous IDs (HN0001, HN0002, ...) from the mapping CSV.
    All other columns remain unchanged.

    Args:
        input_csv:   Path to the original CSV file to be anonymized
        output_csv:  Path where the anonymized CSV will be saved
        mapping_csv: Path to patient_mapping.csv generated by DICOMAnonymizer
        id_column:   Exact name of the column containing original patient IDs
    """
    input_path   = Path(input_csv)
    output_path  = Path(output_csv)
    mapping_path = Path(mapping_csv)

    # ── Validate input files ──────────────────────────────────────────────
    if not input_path.exists():
        print(f"ERROR: Input CSV not found at: {input_csv}")
        return

    if not mapping_path.exists():
        print(f"ERROR: Mapping CSV not found at: {mapping_csv}")
        print(f"  Run DICOMAnonymizer.anonymize_study() first to generate it.")
        return

    # ── Load files ────────────────────────────────────────────────────────
    data_df    = pd.read_csv(input_path)
    mapping_df = pd.read_csv(mapping_path)

    print(f"\nLoaded input CSV:   {input_csv}  ({len(data_df)} rows, {len(data_df.columns)} columns)")
    print(f"Loaded mapping CSV: {mapping_csv}  ({len(mapping_df)} patients)")

    # ── Validate ID column exists ─────────────────────────────────────────
    if id_column not in data_df.columns:
        print(f"\nERROR: Column '{id_column}' not found in input CSV.")
        print(f"  Available columns: {list(data_df.columns)}")
        return

    # ── Validate mapping CSV has required columns ─────────────────────────
    required_cols = {'original_patient_id', 'anonymous_id'}
    if not required_cols.issubset(mapping_df.columns):
        print(f"\nERROR: Mapping CSV must contain columns: {required_cols}")
        print(f"  Found columns: {set(mapping_df.columns)}")
        return

    # ── Build lookup: original_patient_id -> anonymous_id ─────────────────
    # Strip whitespace to avoid silent mismatches
    id_map = {
        str(row['original_patient_id']).strip(): str(row['anonymous_id']).strip()
        for _, row in mapping_df.iterrows()
    }

    # ── Replace IDs ───────────────────────────────────────────────────────
    original_ids  = data_df[id_column].astype(str).str.strip()
    anonymized    = original_ids.map(id_map)

    # Track which IDs had no mapping entry
    not_found_mask = anonymized.isna()
    not_found_ids  = original_ids[not_found_mask].unique().tolist()

    if not_found_ids:
        print(f"\nWARNING: {len(not_found_ids)} ID(s) in '{id_column}' "
              f"had no match in the mapping CSV — these rows are left unchanged:")
        for pid in not_found_ids:
            print(f"    - {pid}")

    # Fill unmatched rows with original value (leave unchanged)
    data_df[id_column] = anonymized.where(~not_found_mask, other=original_ids)

    # ── Save anonymized CSV ───────────────────────────────────────────────
    output_path.parent.mkdir(parents=True, exist_ok=True)
    data_df.to_csv(output_path, index=False)

    # ── Summary ───────────────────────────────────────────────────────────
    success_count = (~not_found_mask).sum()
    print(f"\n========================================")
    print(f"CSV Anonymization Complete!")
    print(f"  Column anonymized:       '{id_column}'")
    print(f"  IDs successfully mapped: {success_count}/{len(data_df)} rows")
    print(f"  IDs with no mapping:     {not_found_mask.sum()}/{len(data_df)} rows")
    print(f"  Columns unchanged:       {[c for c in data_df.columns if c != id_column]}")
    print(f"  Output saved to:         {output_csv}")
    print(f"========================================")

    # ── Preview ───────────────────────────────────────────────────────────
    print(f"\nFirst 5 rows of anonymized CSV:")
    print(data_df.head())


# ── USAGE ─────────────────────────────────────────────────────────────────────
anonymize_csv_id_column(
    input_csv   = 'data/pre-processed/cmc.csv',
    output_csv  = 'data/pre-processed/anonymized_cmc.csv',
    mapping_csv = 'anonymized_study/patient_mapping.csv',
    id_column   = 'id'    # Change this to match your exact column name
)


Loaded input CSV:   data/pre-processed/cmc.csv  (102 rows, 12 columns)
Loaded mapping CSV: anonymized_study/patient_mapping.csv  (102 patients)

CSV Anonymization Complete!
  Column anonymized:       'id'
  IDs successfully mapped: 102/102 rows
  IDs with no mapping:     0/102 rows
  Columns unchanged:       ['site', 'sex', 'overall_hpv_status', 'tstage', 'nstage', 'mstage', 'groupstage', 'event_locoregional_recurrence', 'locoregional_recurrence_in_days', 'vol', 'area']
  Output saved to:         data/pre-processed/anonymized_cmc.csv

First 5 rows of anonymized CSV:
       id         site   sex overall_hpv_status  tstage  nstage  mstage  \
0  HN0001       larynx  male                NaN       3       0       0   
1  HN0004  hypopharynx  male                NaN       4       3       0   
2  HN0006  nasopharynx  male           negative       1       0       0   
3  HN0091       larynx  male                NaN       3       0       0   
4  HN0015  hypopharynx  male                NaN    